In [ ]:
from valdpy import ValdAuth, ForeDecksAPI
from valdpy.utils import read_credentials

%load_ext autoreload
%autoreload 2

# ForceDecks API Example

This example demonstrates how to use the VALDPY package to access ForceDecks (force plate) test data.

## Prerequisites

1. Have VALDPY installed: `pip install -e ..`
2. Create a credentials file: Copy `vald_api_cred_TEMPLATE.txt` to `vald_api_cred.txt` and fill in your credentials
3. Ensure your VALD tenant has access to ForceDecks data

In [ ]:
# Read credentials from file
creds = read_credentials('vald_api_cred.txt')

client_id = creds['client_id']
client_secret = creds['client_secret']
tenant_id = creds['tenant_id']

print(f"Client ID: {client_id[:10]}...")
print(f"Tenant ID: {tenant_id}")

## Step 1: Authentication

Initialize the ValdAuth class to handle OAuth 2.0 authentication and manage tenant/profile information.

In [ ]:
# Initialize authentication object
auth = ValdAuth(client_id, client_secret, tenant_id=tenant_id, region='USA')

### Get OAuth Access Token

Call `get_token()` to obtain an access token. This should be your first API call.

In [ ]:
# Get access token
token = auth.get_token()
print(f"Token obtained successfully: {token[:20]}...")

### Optional: Get Tenant Information

Retrieve information about your tenant, including available categories and groups.

In [ ]:
# Get all tenants accessible with your credentials
all_tenants = auth.get_all_tenants()
print(f"Found {len(all_tenants)} tenant(s)")
for tenant in all_tenants[:3]:
    print(f"  - {tenant['name']} ({tenant['id']})")

In [ ]:
# Get tenant info
tenant_info = auth.get_tenant_info()
print(f"Tenant Name: {tenant_info.get('name')}")
print(f"Tenant ID: {tenant_info.get('id')}")

## Step 2: Get Categories and Groups

Retrieve available categories and groups to organize your profiles.

In [ ]:
# Get available categories
categories_df = auth.get_tenant_categories()
print("Available Categories:")
print(categories_df[['name', 'id']].to_string(index=False))

In [ ]:
# Get groups for the tenant
groups_df = auth.get_tenant_groups()
print(f"Found {len(groups_df)} group(s)")
print(groups_df[['name', 'id']].head(10).to_string(index=False))

## Step 3: Get Profiles

Retrieve athlete profiles from a specific group.

In [ ]:
# Get profiles from a specific group
# Replace 'Research' with your desired group name
group_name = 'Volleyball'
category_name = 'Team'

try:
    profiles_df = auth.get_group_profiles(groupName=group_name, categoryName=category_name)
    print(f"Found {len(profiles_df)} profile(s) in group '{group_name}'")
    print(profiles_df[['givenName', 'familyName', 'profileId']].head(10).to_string(index=False))
except Exception as e:
    print(f"Error retrieving profiles: {e}")
    print("\nAvailable groups:")
    print(groups_df[['name']].drop_duplicates().to_string(index=False))

In [ ]:
profiles_df

## Step 4: Initialize ForceDecks API

Now that we have authentication and profile information, initialize the ForceDecks API client.

In [ ]:
# Initialize ForceDecks API client
fd = ForeDecksAPI(tenant_id=auth.tenant_id, header=auth.headers, region='USA')

### Get Test Information

Query ForceDecks tests from a specific date. You can optionally filter by profile ID.

In [ ]:
len(profiles_df)

In [ ]:
# Get tests from a specific date
date = '06/08/2026 09:00'  # Format: dd/mm/yyyy

# Optional: filter by profile ID
if len(profiles_df) > 0:
    print(f"Using profile ID: {profiles_df.iloc[0]['profileId']}")
    profile_id = profiles_df.iloc[2]['profileId']
    tests_df = fd.get_tests_info(date, profile_id=profile_id)
else:
    tests_df = fd.get_tests_info(date)

if tests_df is not None:
    print(f"Found {len(tests_df)} test(s) from {date}")
else:
    print(f"No tests found on {date}")

In [ ]:
# View test information
if tests_df is not None and len(tests_df) > 0:
    print(tests_df[['testId', 'profileId', 'modifiedDateUtc']].head())
else:
    print("No test data available. Try a different date range.")

### Get Test Results

Retrieve detailed results (metrics for each repetition) from a specific test.

In [ ]:
# Get results from the first test
if tests_df is not None and len(tests_df) > 0:
    test_id = tests_df.iloc[0]['testId']
    print(f"Retrieving results for test: {test_id}")
    
    response_json, result_defs = fd.get_test_results(test_id)
    print(f"Found {len(fd.results_df)} result(s)")
    print(fd.results_df.head())
else:
    print("No tests available to retrieve results from")

### Get Raw Force Trace

Retrieve the raw force-time data for a test.

In [ ]:
# Get force trace (raw data)
if tests_df is not None and len(tests_df) > 0:
    test_id = tests_df.iloc[0]['testId']
    force_trace = fd.get_force_trace(test_id)
    
    if force_trace is not None:
        print(f"Force trace shape: {force_trace.shape}")
        print(force_trace.head())
    else:
        print("No force trace data available")
else:
    print("No tests available")

In [ ]:
# Visualize force trace (if available)
if force_trace is not None and 'Time' in force_trace.columns:
    import matplotlib.pyplot as plt
    
    plt.figure(figsize=(12, 5))
    if 'Right' in force_trace.columns:
        plt.plot(force_trace['Time'], force_trace['Right'], label='Right')
    if 'Left' in force_trace.columns:
        plt.plot(force_trace['Time'], force_trace['Left'], label='Left')
    
    plt.xlabel('Time')
    plt.ylabel('Force')
    plt.title('Force Trace')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print("Force trace data not available for visualization")